# Lecture 13 – Visualizations II

### DATA 2201, Fall 2026



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

## Lecture Outlines
- #### Kernel Density Estimate (KDE)
      - KDE Algorithm & Examples
      - Effect of bandwidth hyperparameter
- #### Plotting Distributions - Revisited
- #### Scatter Plots
- #### Hex Plots
- #### Contour Plots 

### Demo Dataset: World Bank Population

In [ ]:
wb = pd.read_csv("data/world_bank.csv", index_col=0)
wb = wb.rename(columns={'Antiretroviral therapy coverage: % of people living with HIV: 2015':"HIV rate",
                       'Gross national income per capita, Atlas method: $: 2016':'gni'})
wb.head()

## Kernel Density Estimate (KDE)

A KDE is a smooth curve that helps us see the overall shape of a distribution.

**Main idea:** Put one small bump at each data point, add the bumps, and divide by the number of points.

We will build the curve directly, one step at a time, without helper functions.


### Step 1 — Start with five data points

The short marks at the bottom (a **rug plot**) show the original observations. They do not show density.


In [ ]:
points = np.array([2.2, 2.8, 3.7, 5.3, 5.7])
n = len(points)
print("Data points:", points)
print("Number of points:", n)

plt.figure(figsize=(8, 3))
sns.rugplot(x=points, height=0.15)
plt.xlim(0, 8)
plt.xlabel("Value")
plt.title("Our five observations")
plt.show()

### Step 2 — Choose where to draw the curve and how wide each bump is

- `x` contains 1,000 positions along the horizontal axis. These are plotting positions, **not new observations**.
- `h` is the **bandwidth**: the width of each Gaussian bump. Here, `h = 0.5`.
- At every position in `x`, we calculate a height for the curve.


In [ ]:
x = np.linspace(-3, 11, 1000)
h = 0.5
print("First five plotting positions:", x[:5])

### Step 3 — Draw one bump around the first point

The first observation is `2.2`. Its bump is tallest at `2.2` and becomes smaller farther away.

The Gaussian formula below produces all 1,000 heights at once. Each complete bump has area 1; its height is a **density**, not a probability.


In [ ]:
point = points[0]  # 2.2
bump = np.exp(-0.5 * ((x - point) / h) ** 2) / (h * np.sqrt(2 * np.pi))

plt.figure(figsize=(8, 4))
plt.plot(x, bump, label="Bump centered at 2.2")
plt.axvline(point, color="gray", linestyle="--", label="First data point")
plt.xlabel("Value")
plt.ylabel("Density")
plt.title("One Gaussian bump")
plt.legend()
plt.show()

### Step 4 — Draw one bump for every point and add their heights

The loop repeats the same calculation for each of our five observations.

`total` starts at zero. Each `total = total + bump` adds one bump's heights to the running total. The dashed black curve is the sum of all five bumps; its total area is 5.


In [ ]:
total = np.zeros(len(x))

plt.figure(figsize=(8, 4))
for point in points:
    bump = np.exp(-0.5 * ((x - point) / h) ** 2) / (h * np.sqrt(2 * np.pi))
    total = total + bump
    plt.plot(x, bump, label=f"Bump at {point}")

plt.plot(x, total, color="black", linestyle="--", linewidth=2, label="Sum of bumps")
plt.xlabel("Value")
plt.ylabel("Height")
plt.title("Five bumps and their sum")
plt.legend()
plt.show()


### Step 5 — Divide by the number of points: this is the KDE

We have five bumps, each with area 1. Dividing their sum by 5 gives a curve with total area 1.

**At any x-position: KDE height = (height of bump 1 + … + height of bump 5) / 5.**

Where observations are close together, their bumps overlap and create a higher density. Probability over an interval is the **area under the curve** over that interval.


In [ ]:
kde = total / n

plt.figure(figsize=(8, 4))
plt.plot(x, kde, color="black", linewidth=2, label="KDE = sum / 5")
sns.rugplot(x=points, height=0.08)
plt.xlabel("Value")
plt.ylabel("Density")
plt.title("Our completed KDE")
plt.legend()
plt.show()


### Step 6 — Check one position numerically

At `x = 3.0`, calculate the height contributed by each point, then average those five heights. This gives the KDE height at that one position.


In [ ]:
x_position = 3.0
heights = np.exp(-0.5 * ((x_position - points) / h) ** 2) / (h * np.sqrt(2 * np.pi))

display(pd.DataFrame({"Data point": points, "Bump height at x = 3.0": heights}))
print("Sum of heights:", heights.sum())
print("KDE height at x = 3.0:", heights.sum() / n)


### Step 7 — Effect (Change) of bandwidth

Use the same data and the same steps, changing only `h`.

- **Small `h` (0.2):** Narrow bumps; more local detail and separate peaks.
- **Medium `h` (0.5):** More overlap; a smoother curve.
- **Large `h` (1.5):** Wide bumps; nearby peaks merge and details may disappear.

Bandwidth changes the estimated shape. The observations stay the same.


In [ ]:
plt.figure(figsize=(8, 4))
for h in [0.2, 0.5, 1.5]:
    total = np.zeros(len(x))
    for point in points:
        bump = np.exp(-0.5 * ((x - point) / h) ** 2) / (h * np.sqrt(2 * np.pi))
        total = total + bump
    kde = total / n
    plt.plot(x, kde, label=f"h = {h}")

sns.rugplot(x=points, height=0.06)
plt.xlabel("Value")
plt.ylabel("Density")
plt.title("Same data, different bandwidths")
plt.legend()
plt.show()


### Same KDE Plot with Seaborn

Once we understand the steps, Seaborn can estimate and plot a Gaussian KDE for us. `kde=True` adds it to a density histogram.

Seaborn selects its own bandwidth by default. `bw_adjust` multiplies that default bandwidth; it is **not** the absolute `h` used above.


In [ ]:
sample_std = np.std(points, ddof=1)

plt.figure(figsize=(8, 4))

for h in [0.2, 0.5, 1.5]:
    sns.kdeplot(
        x=points,
        bw_method=h / sample_std,
        gridsize=1000,
        cut=5,
        label=f"h = {h}"
    )

sns.rugplot(x=points, height=0.06)
plt.xlim(x.min(), x.max())
plt.xlabel("Value")
plt.ylabel("Density")
plt.title("Same data, different bandwidths")
plt.legend()
plt.show()

### Another kernel: a boxcar (rectangle)

A kernel does not have to be a Gaussian bump. A boxcar puts a rectangle around each point.

Here, `h` is the rectangle's **full width**. Its height is `1 / h`, so each rectangle has area 1. We again add the kernels and divide by the number of points. The resulting KDE is less smooth.


In [ ]:
h = 1.0
total = np.zeros(len(x))
for point in points:
    rectangle = np.where(np.abs(x - point) <= h / 2, 1 / h, 0)
    total = total + rectangle

boxcar_kde = total / n
plt.figure(figsize=(8, 4))
plt.plot(x, boxcar_kde)
sns.rugplot(x=points, height=0.08)
plt.xlabel("Value")
plt.ylabel("Density")
plt.title("Boxcar KDE: add rectangles, then divide by 5")
plt.show()

### Apply the same steps to World Bank HIV rates

First inspect the histogram and the original observations. Missing values are removed before estimating density.


In [ ]:
vals = wb['HIV rate'].dropna()
ax = sns.histplot(vals)
sns.rugplot(vals, color='orange', ax=ax);

Now compare four bandwidths. Each panel uses the same direct calculation as our five-point example.

Here, `h` is measured in the same units as the HIV-rate values. A very small bandwidth can make the curve sensitive to individual observations; a very large one can hide structure.


In [ ]:
x_hiv = np.linspace(vals.min() - 30, vals.max() + 30, 2000)
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)

for ax, h in zip(axes.flat, [0.1, 1, 2, 5]):
    total = np.zeros(len(x_hiv))
    for point in vals:
        bump = np.exp(-0.5 * ((x_hiv - point) / h) ** 2) / (h * np.sqrt(2 * np.pi))
        total = total + bump
    kde_hiv = total / len(vals)
    ax.plot(x_hiv, kde_hiv)
    ax.set_title(f"HIV rates: h = {h}")
    ax.set_xlabel("HIV rate")
    ax.set_ylabel("Density")

plt.tight_layout()
plt.show()

### Connect the steps to the KDE formula

$$\widehat{f}_h(x) = \frac{1}{n}\sum_{i=1}^{n} K_h(x, x_i)$$

- $x$: the position where we want a density height.
- $x_i$: one observed data point.
- $K_h(x, x_i)$: the height of the bump centered at that point.
- $\sum$: add the heights from all points (`total`).
- $1/n$: divide by the number of points (`kde = total / n`).

**Remember: one bump per observation → add their heights → divide by the number of observations.**


# Plotting Distributions - Revisited

## Diving into `displot`

Seaborn [documentation](https://seaborn.pydata.org/generated/seaborn.displot.html) for `sns.displot` lets you specify the `kind` of plot.

When plotting a histogram,  you can pass in `histplot` ([link](https://seaborn.pydata.org/generated/seaborn.histplot.html#seaborn.histplot)) parameters to `displot` to specify histogram-specific features.

For example, `stat=density` normalizes the histogram such that the area under the histogram is 1.

In [ ]:
sns.displot(data=wb, 
            x="gni", 
            kind="hist", 
            stat="density") # default: stat=count and density integrates to 1
plt.title("Distribution of gross national income per capita");

What does it mean to specify `kind=kde`? We will explore this!

In [ ]:
sns.displot(data=wb, 
            x="gni", 
            kind='kde')
plt.title("Distribution of gross national income per capita");

In [ ]:
sns.displot(data=wb, 
            x="gni", 
            kind='ecdf')
plt.title("Cumulative Distribution of gross national income per capita");

## Scatter Plots

Scatter plots are used to visualize the **relationship** between two **quantitative continuous variables**.

In [ ]:
plt.scatter(wb['per capita: % growth: 2016'], 
            wb['Adult literacy rate: Female: % ages 15 and older: 2005-14'])
plt.xlabel("% growth per capita")
plt.ylabel("Female adult literacy rate");

In [ ]:
sns.scatterplot(data=wb, x='per capita: % growth: 2016', \
                y='Adult literacy rate: Female: % ages 15 and older: 2005-14', 
                hue="Continent")
plt.xlabel("% growth per capita")
plt.ylabel("Female adult literacy rate");

### Observations:
- The above plots suffer from **overplotting** – many scatter points are stacked on top of one another (particularly in the upper right region of the plot). 

**Jittering** is a processed used to address overplotting. A small amount of random noise is added to the x and y values of all datapoints. 

Decreasing the size of each scatter point using the `s` parameter of `plt.scatter` also helps.

In [ ]:
random_x_noise = np.random.uniform(-1, 1, len(wb))
random_y_noise = np.random.uniform(-5, 5, len(wb))

plt.scatter(wb['per capita: % growth: 2016']+random_x_noise, \
            wb['Adult literacy rate: Female: % ages 15 and older: 2005-14']+random_y_noise, s=15)

plt.xlabel("% growth per capita (jittered)")
plt.ylabel("Female adult literacy rate (jittered)");

In [ ]:
sns.lmplot(data=wb, x='per capita: % growth: 2016', \
           y='Adult literacy rate: Female: % ages 15 and older: 2005-14');

In [ ]:
sns.jointplot(data=wb, x='per capita: % growth: 2016', \
              y='Adult literacy rate: Female: % ages 15 and older: 2005-14');

## Hex Plots

- Rather than plot individual datapoints, plot the *density* of how datapoints are distributed in 2D.
- A darker hexagon means that more datapoints lie in that region.

In [ ]:
sns.jointplot(data=wb, x='per capita: % growth: 2016', \
              y='Adult literacy rate: Female: % ages 15 and older: 2005-14',
              kind='hex');

## Contour Plots

- Contour plots are similar to topographic maps.
- Contour lines of the same color have the same *density* of datapoints.
- The region with the darkest color contains the most datapoints of all regions.
- We can think of a contour plot as the 2D equivalent of a KDE curve.

In [ ]:
sns.kdeplot(data=wb, x='per capita: % growth: 2016', \
              y='Adult literacy rate: Female: % ages 15 and older: 2005-14', 
            fill=True);

In [ ]:
sns.jointplot(data=wb, x='per capita: % growth: 2016', \
              y='Adult literacy rate: Female: % ages 15 and older: 2005-14',
              kind='kde');

### Modeling Income and Literacy

We model the relationship between income and female adult literacy by fitting a straight line to **log(income)** and **literacy⁴**:

$$\text{literacy}^4 = m\log(\text{income}) + b$$

We then take the fourth root to return predictions to the original literacy scale. The scatter points show observed values, and the red curve shows the fitted relationship.

In [ ]:
# Select and clean the two columns
df = wb[["gni", "Adult literacy rate: Female: % ages 15 and older: 2005-14"]].copy()
df.columns = ["inc", "lit"]
df = df.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
df = df[(df["inc"] > 0) & (df["lit"] >= 0)].sort_values("inc")

# Fit literacy⁴ against log(income), then transform back
m, b = np.polyfit(np.log(df["inc"]), df["lit"] ** 4, 1)
x = np.linspace(df["inc"].min(), df["inc"].max(), 500)
y = m * np.log(x) + b
y = np.where(y >= 0, y, np.nan) ** 0.25

plt.scatter(df["inc"], df["lit"], label="Observed data")
plt.plot(x, y, color="red", label="Modeled relationship")
plt.xlabel("Gross national income per capita")
plt.ylabel("Female adult literacy rate (%)")
plt.legend();

# See you next lecture!